In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import pandas as pd

from Module.panel_utils import (
    ModelResultsAggregator,
    run_panel_regressions,
    run_spec_tests,
    run_panel_model_diagnostics,
)


In [45]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('reg_analys.xlsx')
df_fed_analys = pd.read_excel('fed_analys.xlsx')


# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])

# Разделим d_Mon_Shock на позитивный и негативный
df_reg_analys['d_Mon_Shock_neg'] = df_reg_analys['d_Mon_Shock'].where(df_reg_analys['d_Mon_Shock'] < 0, 0)
df_reg_analys['d_Mon_Shock_pos'] = df_reg_analys['d_Mon_Shock'].where(df_reg_analys['d_Mon_Shock'] > 0, 0)

# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col != 'Region'
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)


# Взаимодействия (если требуется)
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock'] * df_reg['Covid_dum']

# Кластерные выборки
if 'Cluster_1' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_1'] == 1].copy()
    df_reg_clus_two = df_reg[df_reg['Cluster_2'] == 1].copy()
    df_reg_clus_three = df_reg[(df_reg['Cluster_1'] == 0) & (df_reg['Cluster_2'] == 0)].copy()


In [46]:
###############################
# Построение линейных моделей на панельных данных - общая выборка
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
    'd_Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum'
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6237   R-squared (Within):               0.8925
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:12:01   Log-likelihood                   -9566.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   8.125e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(13,6224)
Min Obs:                       77.000                                           
M

In [47]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
d_Inflation_Expectations  6.821282 0.000000
                exc_rate -6.577659 0.000000
        Int_Rate_FL_lag1 -5.448708 0.000000
              Fin_Dostup -5.122417 0.000000
             d_Mon_Shock -3.425047 0.000619
               Covid_dum  2.752344 0.005934
   Bonds_Rate_Correct_5Y -2.405466 0.016181
          Credit_impulse  2.017414 0.043695
                Sank_dum  1.423187 0.154732
          Cred_structure -0.888940 0.374070
             D_top5_rozn  0.727809 0.466758
           Def_Zadolg_Fl -0.623020 0.533294
               Cred_nagr  0.049685 0.960375

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: sta

In [48]:
# Сохраняем результаты модели для экспорта
pooled_res_1 = pooled_res if pooled_success else None
fe_res_1 = fe_res if fe_success else None
re_res_1 = re_res if re_success else None


In [49]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 55.2831
p-значение: 0.000000
df: 13

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3085.1721
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -69.5429
p-значение: 1.000000
df: (80, 6143)

Вывод: p >= 0.05 - Pooled адекватна


(55.28308793485065,
 3.603903132587405e-07,
 3085.17210891286,
 0.0,
 -69.54288238983575,
 1.0)

In [50]:
###############################
# Построение линейных моделей на панельных данных - общая выборка
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
    'd_Mon_Shock',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'd_Mon_Shock_Covid'
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6237   R-squared (Within):               0.8925
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:12:03   Log-likelihood                   -9565.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   7.545e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(14,6223)
Min Obs:                       77.000                                           
M

In [51]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
d_Inflation_Expectations  6.776560 0.000000
                exc_rate -6.517202 0.000000
        Int_Rate_FL_lag1 -5.436114 0.000000
              Fin_Dostup -5.102594 0.000000
             d_Mon_Shock -3.247647 0.001170
               Covid_dum  2.699152 0.006970
   Bonds_Rate_Correct_5Y -2.381570 0.017269
          Credit_impulse  1.993077 0.046297
                Sank_dum  1.406915 0.159503
          Cred_structure -0.880831 0.378443
             D_top5_rozn  0.742185 0.458003
           Def_Zadolg_Fl -0.610921 0.541274
       d_Mon_Shock_Covid  0.385913 0.699574
               Cred_nagr  0.066242 0.947187

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------


In [52]:
# Сохраняем результаты модели для экспорта
pooled_res_2 = pooled_res if pooled_success else None
fe_res_2 = fe_res if fe_success else None
re_res_2 = re_res if re_success else None


In [53]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 37.9677
p-значение: 0.000526
df: 14

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3085.2369
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -69.6057
p-значение: 1.000000
df: (80, 6142)

Вывод: p >= 0.05 - Pooled адекватна


(37.9676783294506,
 0.0005256050586729399,
 3085.236902004145,
 0.0,
 -69.60569473132271,
 1.0)

In [54]:
###############################
# Построение линейных моделей на панельных данных - общая выборка
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
    'd_Mon_Shock',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'd_Mon_Shock_Cl1',
    'd_Mon_Shock_Cl2'
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6237   R-squared (Within):               0.8926
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:12:05   Log-likelihood                   -9563.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   7.045e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(15,6222)
Min Obs:                       77.000                                           
M

In [55]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
d_Inflation_Expectations  6.807966 0.000000
                exc_rate -6.569839 0.000000
        Int_Rate_FL_lag1 -5.438862 0.000000
              Fin_Dostup -5.121057 0.000000
             d_Mon_Shock -4.098250 0.000042
               Covid_dum  2.749406 0.005988
   Bonds_Rate_Correct_5Y -2.399253 0.016458
          Credit_impulse  2.009803 0.044495
         d_Mon_Shock_Cl1  2.002177 0.045309
         d_Mon_Shock_Cl2  1.477237 0.139663
                Sank_dum  1.421992 0.155079
          Cred_structure -0.890617 0.373169
             D_top5_rozn  0.730119 0.465345
           Def_Zadolg_Fl -0.623075 0.533258
               Cred_nagr  0.054549 0.956500

----------------------------------------------------------------------
MODEL: Pooled OLS
---------------------------

In [56]:
# Сохраняем результаты модели для экспорта
pooled_res_3 = pooled_res if pooled_success else None
fe_res_3 = fe_res if fe_success else None
re_res_3 = re_res if re_success else None


In [57]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 33.5125
p-значение: 0.003984
df: 15

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3085.2820
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -69.5078
p-значение: 1.000000
df: (80, 6141)

Вывод: p >= 0.05 - Pooled адекватна


(33.51246034738099,
 0.003984475288376088,
 3085.2819835127484,
 0.0,
 -69.50781265327703,
 1.0)

In [58]:
###############################
# Построение линейных моделей на панельных данных - кластер 1
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
   # 'ln_New_Loans_Progr',
    'd_Mon_Shock',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_one, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_one'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9940
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                4312   R-squared (Within):               0.8877
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9940
Time:                        15:12:08   Log-likelihood                   -6695.9
Cov. Estimator:                Robust                                           
                                        F-statistic:                   6.511e+04
Entities:                          56   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(11,4301)
Min Obs:                       77.000                                           
Max O

In [59]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
                exc_rate -7.863760 0.000000
d_Inflation_Expectations  7.134254 0.000000
        Int_Rate_FL_lag1 -6.197851 0.000000
           Def_Zadolg_Fl  5.660673 0.000000
          Cred_structure  5.568508 0.000000
             D_top5_rozn  5.032081 0.000001
              Fin_Dostup  4.952358 0.000001
          Credit_impulse  3.848707 0.000120
               Cred_nagr  3.018890 0.002552
             d_Mon_Shock -2.487790 0.012892
   Bonds_Rate_Correct_5Y -0.745263 0.456154

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=457.4825, p=0.000000
White:         stat=1026.8198, p=0.000000

Normality tests
Jarque

In [60]:
# Сохраняем результаты модели для экспорта
pooled_res_4 = pooled_res if pooled_success else None
fe_res_4 = fe_res if fe_success else None
re_res_4 = re_res if re_success else None


In [61]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 74.9469
p-значение: 0.000000
df: 11

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 2127.7751
p-значение: 0.000000
N (регионов): 56, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -74.9228
p-значение: 1.000000
df: (55, 4245)

Вывод: p >= 0.05 - Pooled адекватна


(74.94688060751879,
 1.3876122473277519e-11,
 2127.775060134784,
 0.0,
 -74.92284585978405,
 1.0)

In [62]:
###############################
# Построение линейных моделей на панельных данных - кластер 2
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
   # 'ln_New_Loans_Progr',
    'd_Mon_Shock',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_two, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_two'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9912
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                 462   R-squared (Within):               0.8720
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9912
Time:                        15:12:09   Log-likelihood                   -804.70
Cov. Estimator:                Robust                                           
                                        F-statistic:                      4608.2
Entities:                           6   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                  F(11,451)
Min Obs:                       77.000                                           
Max O

In [63]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
        Int_Rate_FL_lag1 -3.178813 0.001581
               Cred_nagr -2.604169 0.009514
   Bonds_Rate_Correct_5Y -2.558534 0.010838
          Cred_structure  1.591195 0.112268
          Credit_impulse  0.935770 0.349893
             D_top5_rozn -0.865022 0.387488
              Fin_Dostup  0.606053 0.544785
           Def_Zadolg_Fl -0.444123 0.657167
                exc_rate -0.304706 0.760731
             d_Mon_Shock  0.160933 0.872219
d_Inflation_Expectations -0.008865 0.992931

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=43.8248, p=0.000008
White:         stat=213.8995, p=0.000000

Normality tests
Jarque-B

In [64]:
# Сохраняем результаты модели для экспорта
pooled_res_5 = pooled_res if pooled_success else None
fe_res_5 = fe_res if fe_success else None
re_res_5 = re_res if re_success else None


In [65]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 22.0625
p-значение: 0.023894
df: 11

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 230.0611
p-значение: 0.000000
N (регионов): 6, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -88.7865
p-значение: 1.000000
df: (5, 445)

Вывод: p >= 0.05 - Pooled адекватна


(22.062485725382004,
 0.02389360254158268,
 230.06105901981076,
 0.0,
 -88.78651171263817,
 1.0)

In [66]:
###############################
# Построение линейных моделей на панельных данных - кластер 3
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
   # 'ln_New_Loans_Progr',
    'd_Mon_Shock',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations'                               
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_three, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_three'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9944
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                1463   R-squared (Within):               0.8959
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9944
Time:                        15:12:10   Log-likelihood                   -2169.9
Cov. Estimator:                Robust                                           
                                        F-statistic:                   2.332e+04
Entities:                          19   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(11,1452)
Min Obs:                       77.000                                           
Max O

In [67]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
                exc_rate -6.406221 0.000000
        Int_Rate_FL_lag1 -5.291398 0.000000
d_Inflation_Expectations  4.548332 0.000006
             D_top5_rozn  4.377516 0.000013
             d_Mon_Shock -4.368851 0.000013
               Cred_nagr  1.886854 0.059380
           Def_Zadolg_Fl  0.785453 0.432316
   Bonds_Rate_Correct_5Y -0.475576 0.634448
              Fin_Dostup -0.377398 0.705933
          Credit_impulse  0.324350 0.745720
          Cred_structure  0.057007 0.954547

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=191.1100, p=0.000000
White:         stat=537.4458, p=0.000000

Normality tests
Jarque-

In [68]:
# Сохраняем результаты модели для экспорта
pooled_res_6 = pooled_res if pooled_success else None
fe_res_6 = fe_res if fe_success else None
re_res_6 = re_res if re_success else None


In [69]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 14.0780
p-значение: 0.228707
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 719.6435
p-значение: 0.000000
N (регионов): 19, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -56.9181
p-значение: 1.000000
df: (18, 1433)

Вывод: p >= 0.05 - Pooled адекватна


(14.078006204031947,
 0.22870709558056546,
 719.6434795228569,
 0.0,
 -56.91806860134372,
 1.0)

In [70]:
###############################
# Построение линейных моделей на панельных данных - общая выборка ROISFIX
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
   # 'ln_New_Loans_Progr',
    'ROISFIX',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА) - ROISFIX
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9944
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6317   R-squared (Within):               0.8959
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9944
Time:                        15:12:11   Log-likelihood                   -9563.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   1.011e+05
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.988   Distribution:                 F(11,6306)
Min Obs:                       77.000                                   

In [71]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
              Fin_Dostup -9.029836 0.000000
d_Inflation_Expectations  9.268212 0.000000
                exc_rate -7.949080 0.000000
        Int_Rate_FL_lag1 -7.746353 0.000000
          Cred_structure -4.923435 0.000001
                 ROISFIX  4.879269 0.000001
           Def_Zadolg_Fl -4.549237 0.000005
          Credit_impulse  3.840558 0.000124
   Bonds_Rate_Correct_5Y  3.613577 0.000304
             D_top5_rozn -1.913684 0.055706
               Cred_nagr -1.284894 0.198876

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=975.1631, p=0.000000
White:         stat=2169.9010, p=0.000000

Normality tests
Jarque

In [72]:
# Сохраняем результаты модели для экспорта
pooled_res_7 = pooled_res if pooled_success else None
fe_res_7 = fe_res if fe_success else None
re_res_7 = re_res if re_success else None


In [73]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 17.1467
p-значение: 0.103610
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3058.3230
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -69.7229
p-значение: 1.000000
df: (80, 6226)

Вывод: p >= 0.05 - Pooled адекватна


(17.146725654623378,
 0.10360976888179863,
 3058.3230296440024,
 0.0,
 -69.72291927789108,
 1.0)

In [74]:
###############################
# Построение линейных моделей на панельных данных - кластер 1 ROISFIX
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
   # 'ln_New_Loans_Progr',
    'ROISFIX',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_one, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_one'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1 ROISFIX)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9945
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                4367   R-squared (Within):               0.8966
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9945
Time:                        15:12:13   Log-likelihood                   -6582.9
Cov. Estimator:                Robust                                           
                                        F-statistic:                   7.212e+04
Entities:                          56   P-value                           0.0000
Avg Obs:                       77.982   Distribution:                 F(11,4356)
Min Obs:                       77.000                                         

In [75]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
        Int_Rate_FL_lag1 -7.802186 0.000000
d_Inflation_Expectations  6.762068 0.000000
                exc_rate -5.755553 0.000000
          Cred_structure -3.690059 0.000227
                 ROISFIX  3.480256 0.000506
           Def_Zadolg_Fl -3.122221 0.001807
              Fin_Dostup -2.927518 0.003434
   Bonds_Rate_Correct_5Y  2.683223 0.007319
          Credit_impulse  2.567304 0.010282
             D_top5_rozn -1.345752 0.178453
               Cred_nagr -0.182937 0.854856

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=719.3602, p=0.000000
White:         stat=1467.5610, p=0.000000

Normality tests
Jarque

In [76]:
# Сохраняем результаты модели для экспорта
pooled_res_8 = pooled_res if pooled_success else None
fe_res_8 = fe_res if fe_success else None
re_res_8 = re_res if re_success else None


In [77]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 57.0600
p-значение: 0.000000
df: 11

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 2115.1332
p-значение: 0.000000
N (регионов): 56, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -7.8548
p-значение: 1.000000
df: (55, 4301)

Вывод: p >= 0.05 - Pooled адекватна


(57.06004711667305,
 3.2436572094951543e-08,
 2115.133227696258,
 0.0,
 -7.854844480023898,
 1.0)

In [78]:
###############################
# Построение линейных моделей на панельных данных - кластер 2 ROISFIX
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
   # 'ln_New_Loans_Progr',
    'ROISFIX',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_two, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_two'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2 ROISFIX)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9917
Estimator:                  PooledOLS   R-squared (Between):              1.0000
No. Observations:                 468   R-squared (Within):               0.8781
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9917
Time:                        15:12:14   Log-likelihood                   -800.26
Cov. Estimator:                Robust                                           
                                        F-statistic:                      4966.9
Entities:                           6   P-value                           0.0000
Avg Obs:                       78.000   Distribution:                  F(11,457)
Min Obs:                       78.000                                         

In [79]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
        Int_Rate_FL_lag1 -2.828862 0.004877
               Cred_nagr -2.645928 0.008428
   Bonds_Rate_Correct_5Y -1.861778 0.063278
                 ROISFIX  1.582326 0.114268
             D_top5_rozn -1.359775 0.174573
          Credit_impulse  1.244952 0.213789
           Def_Zadolg_Fl -1.170351 0.242471
          Cred_structure  0.950202 0.342513
              Fin_Dostup  0.928039 0.353878
                exc_rate -0.847311 0.397266
d_Inflation_Expectations  0.072766 0.942025

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=69.7346, p=0.000000
White:         stat=265.7972, p=0.000000

Normality tests
Jarque-B

In [80]:
# Сохраняем результаты модели для экспорта
pooled_res_9 = pooled_res if pooled_success else None
fe_res_9 = fe_res if fe_success else None
re_res_9 = re_res if re_success else None


In [81]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 3.0098
p-значение: 0.990595
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 234.4666
p-значение: 0.000000
N (регионов): 6, T (периодов): 78

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -90.0214
p-значение: 1.000000
df: (5, 451)

Вывод: p >= 0.05 - Pooled адекватна


(3.0098267144577937,
 0.9905953827299891,
 234.46661977102127,
 0.0,
 -90.02144181581214,
 1.0)

In [82]:
###############################
# Построение линейных моделей на панельных данных - кластер 3 ROISFIX
# (Int_Rate_FL зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
   # 'ln_New_Loans_Progr',
    'ROISFIX',
    'ln_New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'exc_rate',
    'd_Inflation_Expectations'                               
]

dependent_var = 'Int_Rate_FL'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_three, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_three'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3 ROISFIX)
Зависимая переменная: Int_Rate_FL

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:            Int_Rate_FL   R-squared:                        0.9949
Estimator:                  PooledOLS   R-squared (Between):              1.0000
No. Observations:                1482   R-squared (Within):               0.9034
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9949
Time:                        15:12:15   Log-likelihood                   -2129.1
Cov. Estimator:                Robust                                           
                                        F-statistic:                    2.59e+04
Entities:                          19   P-value                           0.0000
Avg Obs:                       78.000   Distribution:                 F(11,1471)
Min Obs:                       78.000                                         

In [83]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_FL
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
                exc_rate -5.060700 0.000000
        Int_Rate_FL_lag1 -4.885718 0.000001
              Fin_Dostup -4.525761 0.000007
d_Inflation_Expectations  3.789661 0.000157
          Cred_structure -3.786155 0.000159
           Def_Zadolg_Fl -3.598606 0.000331
          Credit_impulse  3.099787 0.001973
                 ROISFIX  2.768361 0.005705
               Cred_nagr -2.583691 0.009871
   Bonds_Rate_Correct_5Y  1.768324 0.077214
             D_top5_rozn  1.386458 0.165817

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=338.7870, p=0.000000
White:         stat=794.0503, p=0.000000

Normality tests
Jarque-

In [84]:
# Сохраняем результаты модели для экспорта
pooled_res_10 = pooled_res if pooled_success else None
fe_res_10 = fe_res if fe_success else None
re_res_10 = re_res if re_success else None


In [85]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 9.9769
p-значение: 0.532468
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 737.4031
p-значение: 0.000000
N (регионов): 19, T (периодов): 78

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -79.4537
p-значение: 1.000000
df: (18, 1452)

Вывод: p >= 0.05 - Pooled адекватна


(9.976884796987804,
 0.5324675795808578,
 737.403128421045,
 0.0,
 -79.45374069183833,
 1.0)

In [86]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'Int_Rate_FL'
base_name = dep_var_name
if base_name.startswith(''):
    base_name = base_name[2:]
if base_name.endswith(''):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_7,
            'fe': fe_res_7,
            're': re_res_7
        }
    }
]

model_specs_cluster = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_4,
            'fe': fe_res_4,
            're': re_res_4
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_8,
            'fe': fe_res_8,
            're': re_res_8
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_5,
            'fe': fe_res_5,
            're': re_res_5
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_9,
            'fe': fe_res_9,
            're': re_res_9
        }
    },
    {
        'spec_name': 'Модель_5',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_6,
            'fe': fe_res_6,
            're': re_res_6
        }
    },
    {
        'spec_name': 'Модель_6',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_10,
            'fe': fe_res_10,
            're': re_res_10
        }
    }
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_all_data.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

aggregator_cluster = ModelResultsAggregator()
for spec in model_specs_cluster:
    add_model_set(aggregator_cluster, spec)

out_cluster = os.path.join(results_dir, f"{base_name}_cluster_data.xlsx")
build_and_export(aggregator_cluster, out_cluster, include_pvalues=True, decimals=3)


,Модель_1 (POOL),Модель_1 (FE),Модель_1 (RE),Модель_2 (POOL),Модель_2 (FE),Модель_2 (RE),Модель_3 (POOL),Модель_3 (FE),Модель_3 (RE),Модель_4 (POOL),Модель_4 (FE),Модель_4 (RE),Модель_5 (POOL),Модель_5 (FE),Модель_5 (RE),Модель_6 (POOL),Модель_6 (FE),Модель_6 (RE)
Зависимая переменная,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL,Int_Rate_FL
Int_Rate_FL_lag1,0.853*** (0.000),0.777*** (0.000),0.853*** (0.000),0.746*** (0.000),0.701*** (0.000),0.746*** (0.000),0.779*** (0.000),0.682*** (0.000),0.779*** (0.000),0.708*** (0.000),0.630*** (0.000),0.708*** (0.000),0.824*** (0.000),0.750*** (0.000),0.824*** (0.000),0.718*** (0.000),0.648*** (0.000),0.718*** (0.000)
Cred_nagr,0.044 (0.880),-0.663 (0.352),0.044 (0.880),0.474* (0.087),-1.828*** (0.009),0.474* (0.087),-6.504** (0.022),-6.482 (0.100),-6.504** (0.022),-4.759* (0.063),-8.215** (0.039),-4.759* (0.063),0.014 (0.979),-3.473*** (0.002),0.014 (0.979),0.170 (0.735),-5.441*** (0.000),0.170 (0.735)
D_top5_rozn,0.945*** (0.000),8.886*** (0.000),0.945*** (0.000),2.572*** (0.000),4.056*** (0.002),2.572*** (0.000),0.304 (0.839),-30.144*** (0.001),0.304 (0.839),4.055** (0.035),-28.766*** (0.001),4.055** (0.035),3.813*** (0.000),4.211 (0.148),3.813*** (0.000),3.354*** (0.000),-1.584 (0.571),3.354*** (0.000)
Fin_Dostup,-0.009** (0.015),0.010 (0.524),-0.009** (0.015),0.007** (0.048),0.018 (0.202),0.007** (0.048),0.006 (0.726),-0.101** (0.012),0.006 (0.726),-0.001 (0.972),-0.056 (0.198),-0.001 (0.972),-0.042*** (0.000),-0.036 (0.179),-0.042*** (0.000),0.006 (0.481),-0.009 (0.715),0.006 (0.481)
Credit_impulse,-0.011*** (0.001),-0.007** (0.046),-0.011*** (0.001),-0.001 (0.735),0.004 (0.264),-0.001 (0.735),0.006 (0.667),0.005 (0.742),0.006 (0.667),0.022 (0.161),0.019 (0.248),0.022 (0.161),-0.011* (0.064),-0.006 (0.370),-0.011* (0.064),0.000 (0.952),0.012* (0.055),0.000 (0.952)
Cred_structure,1.383*** (0.000),6.239*** (0.000),1.383*** (0.000),-0.322 (0.346),2.009** (0.021),-0.322 (0.346),4.195** (0.035),11.554*** (0.000),4.195** (0.035),-0.911 (0.694),6.374** (0.026),-0.911 (0.694),-0.126 (0.825),4.052* (0.053),-0.126 (0.825),-0.262 (0.632),1.544 (0.435),-0.262 (0.632)
Def_Zadolg_Fl,0.057*** (0.000),0.044 (0.257),0.057*** (0.000),0.124*** (0.000),-0.023 (0.543),0.124*** (0.000),0.186* (0.060),0.561** (0.018),0.186* (0.060),0.138 (0.138),0.188 (0.419),0.138 (0.138),0.034 (0.284),-0.208*** (0.005),0.034 (0.284),0.127*** (0.000),-0.270*** (0.000),0.127*** (0.000)
d_Mon_Shock,-0.035*** (0.000),-0.038*** (0.000),-0.035*** (0.000),,,,0.026 (0.353),0.018 (0.501),0.026 (0.353),,,,-0.049*** (0.001),-0.052*** (0.000),-0.049*** (0.001),,,
Bonds_Rate_Correct_5Y,-0.455*** (0.000),-0.563*** (0.000),-0.455*** (0.000),-0.083** (0.019),-0.221*** (0.000),-0.083** (0.019),-0.779*** (0.000),-1.104*** (0.000),-0.779*** (0.000),-0.353** (0.029),-0.700*** (0.000),-0.353** (0.029),-0.496*** (0.000),-0.686*** (0.000),-0.496*** (0.000),-0.131** (0.027),-0.346*** (0.000),-0.131** (0.027)
